In [1]:
import pandas as pd
import random
import time
pd.set_option('display.max_rows',20)

In [2]:
random.seed(17)

In [3]:
num_activities = 4
max_tries = 3
prob_blink = 0.1

In [4]:
start_time = time.time()

In [5]:
activities = [f'a{i+1}' for i in range(num_activities)]
print(activities)

['a1', 'a2', 'a3', 'a4']


In [6]:
activity_types = {}
for a in activities:
    activity_types[a] = random.sample(['flyby','mower','circle'],1)[0]
print(activity_types)

{'a1': 'circle', 'a2': 'mower', 'a3': 'mower', 'a4': 'mower'}


In [7]:
num_regions = 2**num_activities
coverage = {}
for i in range(num_regions):
    if i>0:
        coverage[f'r{i}'] = [a for j,a in enumerate(activities) if 2**j & i > 0]
regions = [r for r in coverage.keys()]
region_probs = [0.75+random.random() for r in coverage]
region_probs = [p / sum(region_probs) for p in region_probs]
#region_probs = [0,0,0,0,0,0,1] # always vis to all
assert abs(sum(region_probs)-1)<1e-10
print('regions: ',region_probs)
pdf_region = pd.DataFrame({'region': coverage.keys(),
                           'prob': region_probs})
pdf_region['coverage'] = [coverage[r] for r in regions]
pdf_region

regions:  [0.05414156592747125, 0.07895577825656366, 0.0757327999851681, 0.0735019464472059, 0.044795482368756544, 0.04046128031312302, 0.0590653142401742, 0.07792920498055463, 0.052202098942740034, 0.0651809103095121, 0.05558983895944513, 0.083133491321297, 0.08828223612564413, 0.05997250352458498, 0.09105554829775936]


,region,prob,coverage
0,r1,0.054142,[a1]
1,r2,0.078956,[a2]
2,r3,0.075733,"[a1, a2]"
3,r4,0.073502,[a3]
4,r5,0.044795,"[a1, a3]"
5,r6,0.040461,"[a2, a3]"
6,r7,0.059065,"[a1, a2, a3]"
7,r8,0.077929,[a4]
8,r9,0.052202,"[a1, a4]"
9,r10,0.065181,"[a2, a4]"


In [8]:
pdf_app = pd.DataFrame({'appearance':['blend','neutral','obvious'],
                        'prob' : [0.2,0.4,0.4]})
assert sum(pdf_app['prob'])==1
pdf_app

,appearance,prob
0,blend,0.2
1,neutral,0.4
2,obvious,0.4


In [9]:
pdf_occ = pd.DataFrame({'occlusion':['easy','med','hard'],
                        'prob' : [0.5,0.3,0.2]})
assert sum(pdf_occ['prob'])==1
pdf_occ

,occlusion,prob
0,easy,0.5
1,med,0.3
2,hard,0.2


In [10]:
prob_occ_a_given_occlusion = {'circle':{'hard':0.5,'med':0.3,'easy':0.1},
                              'mower':{'hard':0.7,'med':0.5,'easy':0.3},
                              'flyby':{'hard':0.9,'med':0.7,'easy':0.5},}

In [11]:
#prob_occ_a_given_occlusion = {'circle':{'hard':0.,'med':0.,'easy':0.},
#                              'mower':{'hard':0.,'med':0.,'easy':0.},
#                              'flyby':{'hard':0.,'med':0.,'easy':0.},}

In [12]:
prob_indis_a_given_appearance = {'blend':0.9,'neutral':0.5,'obvious':0.1}

In [13]:
#prob_indis_a_given_appearance = {'blend':0.,'neutral':0.,'obvious':0.}

In [14]:
big_table1 = pd.merge(pdf_app,pdf_occ,how='cross',suffixes=('_app','_occ'))
big_table2 = pd.merge(big_table1,pdf_region,how='cross').rename(columns={'prob':'prob_reg'})
big_table2

,appearance,prob_app,occlusion,prob_occ,region,prob_reg,coverage
0,blend,0.2,easy,0.5,r1,0.054142,[a1]
1,blend,0.2,easy,0.5,r2,0.078956,[a2]
2,blend,0.2,easy,0.5,r3,0.075733,"[a1, a2]"
3,blend,0.2,easy,0.5,r4,0.073502,[a3]
4,blend,0.2,easy,0.5,r5,0.044795,"[a1, a3]"
...,...,...,...,...,...,...,...
130,obvious,0.4,hard,0.2,r11,0.055590,"[a1, a2, a4]"
131,obvious,0.4,hard,0.2,r12,0.083133,"[a3, a4]"
132,obvious,0.4,hard,0.2,r13,0.088282,"[a1, a3, a4]"
133,obvious,0.4,hard,0.2,r14,0.059973,"[a2, a3, a4]"


In [15]:
big_table3 = big_table2
for a in activities:
    big_table3 = pd.merge(big_table3,pd.DataFrame({'num_'+a:range(max_tries+1)}),how='cross')
big_table3

,appearance,prob_app,occlusion,prob_occ,region,prob_reg,coverage,num_a1,num_a2,num_a3,num_a4
0,blend,0.2,easy,0.5,r1,0.054142,[a1],0,0,0,0
1,blend,0.2,easy,0.5,r1,0.054142,[a1],0,0,0,1
2,blend,0.2,easy,0.5,r1,0.054142,[a1],0,0,0,2
3,blend,0.2,easy,0.5,r1,0.054142,[a1],0,0,0,3
4,blend,0.2,easy,0.5,r1,0.054142,[a1],0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...
34555,obvious,0.4,hard,0.2,r15,0.091056,"[a1, a2, a3, a4]",3,3,2,3
34556,obvious,0.4,hard,0.2,r15,0.091056,"[a1, a2, a3, a4]",3,3,3,0
34557,obvious,0.4,hard,0.2,r15,0.091056,"[a1, a2, a3, a4]",3,3,3,1
34558,obvious,0.4,hard,0.2,r15,0.091056,"[a1, a2, a3, a4]",3,3,3,2


In [16]:
def prob_det_a(a,r):
    if a not in r['coverage']:
        return 0.0
    if r['num_'+a] <= 0:
        return 0.0
    prob_not_indis_a = r['prob_app']*(1-prob_indis_a_given_appearance[r['appearance']])
    prob_not_occ_a = r['prob_occ']*(1-prob_occ_a_given_occlusion[activity_types[a]][r['occlusion']])
    prob_det = prob_not_indis_a*prob_not_occ_a*r['prob_reg']*(1-prob_blink**r['num_'+a])
    return prob_det

for a in activities:
    big_table3['prob_det_'+a] = big_table3.apply(lambda r: prob_det_a(a,r), axis=1)
big_table3

,appearance,prob_app,occlusion,prob_occ,region,prob_reg,coverage,num_a1,num_a2,num_a3,num_a4,prob_det_a1,prob_det_a2,prob_det_a3,prob_det_a4
0,blend,0.2,easy,0.5,r1,0.054142,[a1],0,0,0,0,0.000000,0.000000,0.000000,0.000000
1,blend,0.2,easy,0.5,r1,0.054142,[a1],0,0,0,1,0.000000,0.000000,0.000000,0.000000
2,blend,0.2,easy,0.5,r1,0.054142,[a1],0,0,0,2,0.000000,0.000000,0.000000,0.000000
3,blend,0.2,easy,0.5,r1,0.054142,[a1],0,0,0,3,0.000000,0.000000,0.000000,0.000000
4,blend,0.2,easy,0.5,r1,0.054142,[a1],0,0,1,0,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34555,obvious,0.4,hard,0.2,r15,0.091056,"[a1, a2, a3, a4]",3,3,2,3,0.003275,0.001965,0.001947,0.001965
34556,obvious,0.4,hard,0.2,r15,0.091056,"[a1, a2, a3, a4]",3,3,3,0,0.003275,0.001965,0.001965,0.000000
34557,obvious,0.4,hard,0.2,r15,0.091056,"[a1, a2, a3, a4]",3,3,3,1,0.003275,0.001965,0.001965,0.001770
34558,obvious,0.4,hard,0.2,r15,0.091056,"[a1, a2, a3, a4]",3,3,3,2,0.003275,0.001965,0.001965,0.001947


In [17]:
big_table3['prob_row'] = big_table3['prob_app']*big_table3['prob_occ']*big_table3['prob_reg']

In [18]:
big_table3['prob_det_in_0'] = 0.0
for n,a in enumerate(activities):
    big_table3[f'prob_det_in_{n+1}'] = big_table3['prob_row']*(big_table3[f'prob_det_in_{n}']/big_table3['prob_row'] + (1-big_table3[f'prob_det_in_{n}']/big_table3['prob_row'])*big_table3['prob_det_'+a]/big_table3['prob_row'])
big_table3 = big_table3.fillna(0.0)
big_table3

,appearance,prob_app,occlusion,prob_occ,region,prob_reg,coverage,num_a1,num_a2,num_a3,...,prob_det_a1,prob_det_a2,prob_det_a3,prob_det_a4,prob_row,prob_det_in_0,prob_det_in_1,prob_det_in_2,prob_det_in_3,prob_det_in_4
0,blend,0.2,easy,0.5,r1,0.054142,[a1],0,0,0,...,0.000000,0.000000,0.000000,0.000000,0.005414,0.0,0.000000,0.000000,0.000000,0.000000
1,blend,0.2,easy,0.5,r1,0.054142,[a1],0,0,0,...,0.000000,0.000000,0.000000,0.000000,0.005414,0.0,0.000000,0.000000,0.000000,0.000000
2,blend,0.2,easy,0.5,r1,0.054142,[a1],0,0,0,...,0.000000,0.000000,0.000000,0.000000,0.005414,0.0,0.000000,0.000000,0.000000,0.000000
3,blend,0.2,easy,0.5,r1,0.054142,[a1],0,0,0,...,0.000000,0.000000,0.000000,0.000000,0.005414,0.0,0.000000,0.000000,0.000000,0.000000
4,blend,0.2,easy,0.5,r1,0.054142,[a1],0,0,1,...,0.000000,0.000000,0.000000,0.000000,0.005414,0.0,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34555,obvious,0.4,hard,0.2,r15,0.091056,"[a1, a2, a3, a4]",3,3,2,...,0.003275,0.001965,0.001947,0.001965,0.007284,0.0,0.003275,0.004356,0.005139,0.005718
34556,obvious,0.4,hard,0.2,r15,0.091056,"[a1, a2, a3, a4]",3,3,3,...,0.003275,0.001965,0.001965,0.000000,0.007284,0.0,0.003275,0.004356,0.005146,0.005146
34557,obvious,0.4,hard,0.2,r15,0.091056,"[a1, a2, a3, a4]",3,3,3,...,0.003275,0.001965,0.001965,0.001770,0.007284,0.0,0.003275,0.004356,0.005146,0.005666
34558,obvious,0.4,hard,0.2,r15,0.091056,"[a1, a2, a3, a4]",3,3,3,...,0.003275,0.001965,0.001965,0.001947,0.007284,0.0,0.003275,0.004356,0.005146,0.005718


In [19]:
#big_table3.to_excel('bt3.xlsx')

In [20]:
columns_num_a = [col for col in big_table3.columns if col.startswith('num_')]
summary_table = big_table3[columns_num_a].drop_duplicates()
def prob_det_nums(r):
    return sum(big_table3[big_table3[columns_num_a].eq(r).all(axis=1)][f'prob_det_in_{num_activities}'])
summary_table['prob_det'] = summary_table.apply(prob_det_nums,axis=1)
summary_table.sort_values('prob_det',ascending=False)

,num_a1,num_a2,num_a3,num_a4,prob_det
255,3,3,3,3,0.529039
239,3,2,3,3,0.528350
251,3,3,2,3,0.528348
254,3,3,3,2,0.528289
191,2,3,3,3,0.528072
...,...,...,...,...,...
32,0,2,0,0,0.169141
1,0,0,0,1,0.167600
4,0,0,1,0,0.157931
16,0,1,0,0,0.153764


In [21]:
end_time = time.time()
print(f'Ran in {end_time - start_time:.1f} seconds')

Ran in 1.0 seconds
